In [ ]:
"""
This script creates artificial data for a data count estimation problem.
Seven variables were designed as significant and three as non-significant.
"""



import numpy as np
import pandas as pd
import random
import scipy.stats as ss
from scipy.stats import maxwell

from statsmodels.discrete.discrete_model import genpoisson_p

import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri

r = robjects.r
r['source']('CONWAY_MODEL_R.R')
pandas2ri.activate()
genpon_function_r = robjects.globalenv['GENUNI']




def noise(n_obs, perc=.2):
    noise_vec = np.zeros(n_obs)
    rand_pos = np.random.randint(n_obs, size=(int(n_obs*perc)))
    noise_vec[rand_pos] = np.random.normal(size=(int(n_obs*perc)))
    return noise_vec


np.random.seed(0)
N = 10000  # Number of observations

# Generate input data
df = pd.DataFrame()
df['id'] = np.arange(1, N+1)


df['const'] = np.ones(N)
df['x1'] = random.choices(range(300, 3000), k = N)
df['x1'] = np.log(df['x1'])
df['x2'] = random.choices([0.1, 0.2, 0.3, 0.4], k = N) + noise(N)
df['x2'] = random.choices([50, 60, 70, 80, 90, 100, 110], weights = [1, 2, 1, 3, 2, 5, 2], k = N)
df['x2'] = np.log(df['x2'])
df['x3'] = random.choices(range(1, 6), k = N)
df['x4'] = random.choices(range(1, 7), k =N)+noise(N)
df['x5'] = random.choices([0, 1], k = N)
df['x6'] = random.choices([1,2,3,4], k = N)
df['x7'] = random.choices(range(0, 2), k = N)


df['non1'] = random.choices(range(1, 100), k = N)+noise(N)
df['non2'] = random.choices(range(1, 50), k = N)+noise(N)
df['non3'] = random.choices(range(20, 30), k = N)+noise(N)
df['non4'] = random.choices(range(4, 9), k = N)+noise(N)
df['non5'] = random.choices(range(0, 2), k = N)
df['non6'] = random.choices(range(0, 2), k = N)
df['non7'] = random.choices(range(0, 2), k = N)
df['non8'] = random.choices(range(0, 2), k = N)
df = df.round(3)

# Define coefficients (betas)
# Fixed betas
Bconst, Baadt, Bcurve, Bspeed, Blanes, Bmedwidth = 2, 2.1, 2.15, 1.3, 0.2, -2

# Random betas
Bdayr = np.random.normal(loc=-4.5, scale=1, size=N)
Bcarggr = np.random.normal(loc=-5, scale=1, size=N)
Bvehicler = np.random.normal(loc=-2.5, scale=1, size=N)

# Convert betas to matrix for easy product
B = [np.repeat(Bconst, N), np.repeat(Baadt, N), np.repeat(Bcurve, N), np.repeat(Bspeed, N), np.repeat(Blanes, N),
     np.repeat(Bmedwidth, N), np.repeat(Bdayr, 1), np.repeat(Bcarggr, 1),
     np.repeat(Bvehicler, 1), np.zeros(N), np.zeros(N), np.zeros(N),  np.zeros(N), np.zeros(N), np.zeros(N), np.zeros(N), np.zeros(N)]
B = np.vstack(B).T

# Multiply and generate probability
X = df.values[:, 2:]  # Extract only necessary columns





XB = (X*B).sum(axis=1).reshape(N, 1)
eXB = np.exp(XB)
eXB = np.array(eXB).transpose()
print(XB)

XB = np.array(XB).transpose()
        
try:
    mean_2 = genpon_function_r(N, eXB, -.1)
  
except Exception as e:
    print(e)
    print('wg')
    
print(mean_2)

mean_2 = [round(x) for x in mean_2 ]
mean_2 = np.array(mean_2)
print('the sum of the means', sum(mean_2), 'the max ', max(mean_2))

df['choice'] = mean_2

# Save to CSV
df.to_csv("artificial.csv", index=False)

In [ ]:
def rpois(n,mu):
    """
    Generates random variables from the Poisson distribution
    """
    from scipy.stats import poisson
    result=poisson.rvs(size=n,mu=mu)
    return result

In [ ]:
def gen_pos():
    genpoisson_p._mean()
    

In [ ]:
def rCMP(lam, nu, tol = 0.01):
    
    #if mean != minvalue:
       # return mean
    print(lam)
    z = 1+ lam
    print(z)
    a1 = lam*lam/np.power(2, nu)
   
    zx = lam
    ax1 = 2*a1
    
    for i in range(3, 100000):
        
        e = lam/np.power(i, nu)
        ex  = lam/np.power(i, nu -1)/(i-1)
        print(i)
        try:
            a2 = a1*e
            ax2 = ax1*ex
            print(a2)
            print(ax2)
            print(a1)
            print(ax1)
        except Exception as e:
            print(e)
        if np.all(ax2) <np.all(ax1) and np.all(a2)<np.all(a1):
            print('yes')
            m = zx/z
            upper = (zx+(ax1/(1-(ax2/ax1))))/z
            lower = zx/(z+(a1/(1-(a2/a1))))
            
            r = (upper-lower)/m
            if np.all(r) < tol:
                break
        
        z =z+a1
        zx=zx+ax1
        a1=a2
        ax1 =ax2
    mean = zx/z
    return mean